In [1]:
import os
from typing import Optional, List, Union, Tuple
import pandas as pd
import numpy as np
import mlflow
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
import evaluate
from tqdm.notebook import tqdm
import torch
import torch.nn as nn
from torch.nn import CrossEntropyLoss, NLLLoss
from torch.utils.data import Dataset, DataLoader
import transformers
from transformers import (
    AutoConfig,
    AutoModel,
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    BertModel,
    BertPreTrainedModel,
    PreTrainedModel, 
    PretrainedConfig,
    TrainingArguments, 
    Trainer, 
    DataCollatorWithPadding,
    TextClassificationPipeline
)
from transformers.modeling_outputs import SequenceClassifierOutput
from transformers.file_utils import add_code_sample_docstrings, add_start_docstrings_to_model_forward
from dataset import MultitaskDataset 

/mnt/DATA/environments/miniconda3/envs/ast-transformer/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
os.environ['WANDB_DISABLED'] = 'true'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
# model_path = 'intfloat/e5-base-v2'
model_path = "google-bert/bert-base-uncased"
cache_dir='../cache'

In [3]:
dataset = pd.read_excel('../data/Misijos_duomenims.xlsx', sheet_name='Komentarai')
data = dataset[['Comment']]
labels = dataset.iloc[:, -6:-1]
labels.columns = ['insult', 'hate', 'illegitimate', 'manipulation_strength', 'manipulation_type']

In [4]:
class MultitaskTextClassifier(BertPreTrainedModel):

    def __init__(self, config, base_model, frozen_layers_count=0, **kwargs):
        super().__init__(config)
        self.num_labels = kwargs.get("task_labels_map", {})
        # self.num_labels = 2
        config = config or PretrainedConfig.from_pretrained(base_model)
        config.problem_type = "single_label_classification"
        config.output_hidden_states = True
        config.use_weighted_layer_sum = False
        self.config = config
        self.model = BertModel.from_pretrained(base_model, config=self.config, cache_dir=cache_dir)
        # Fine-tune full model
        if frozen_layers_count == -1:
            for param in self.model.base_model.parameters():
                param.requires_grad = True            
        else:
            for param in self.model.base_model.parameters():
                param.requires_grad = False
            # Unfreeze last frozen_layers_count layers for finetuning
            if frozen_layers_count > 0:
                for idx in range(-1, -frozen_layers_count-1, -1):
                    for param in self.model.encoder.layers[idx].parameters():
                        param.requires_grad = True
        classifier_dropout = (
            config.classifier_dropout
            if config.classifier_dropout is not None
            else config.hidden_dropout_prob
        )
        self.dropout = nn.Dropout(classifier_dropout)
        ## add task specific output heads
        self.classifiers = nn.ModuleList()
        for key, output_size in self.num_labels.items():
            self.classifiers.append(nn.Linear(config.hidden_size, output_size))
        self.init_weights()


    def forward(self,
        input_ids: Optional[torch.Tensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        token_type_ids: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.Tensor] = None,
        head_mask: Optional[torch.Tensor] = None,
        inputs_embeds: Optional[torch.Tensor] = None,
        labels: Optional[torch.Tensor] = None,
        output_attentions: Optional[bool] = None,
        output_hidden_states: Optional[bool] = None       
    ) -> SequenceClassifierOutput:
        outputs = self.model(
            input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            position_ids=position_ids,
            head_mask=head_mask,
            inputs_embeds=inputs_embeds,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states        
        )
        x = outputs[1]
        x = self.dropout(x)
        outputs = list()
        total_loss = 0
        for ind, classifier in enumerate(self.classifiers):
            logits = classifier(x)
            loss = None
            loss_fct = NLLLoss()
            print(logits.view(-1, self.num_labels[ind]), labels[:, ind])
            loss = loss_fct(logits.view(-1, self.num_labels[ind]), labels[:, ind])
            total_loss = total_loss + loss
            outputs.append(logits)
        logits = torch.cat(outputs)
        outputs.append(SequenceClassifierOutput(loss=total_loss, logits=logits))
        return outputs

In [5]:
def get_training_args(output_dir, num_epochs=20):    
    core_params=dict(
        output_dir=output_dir,
        # metric_for_best_model='accuracy',
        load_best_model_at_end=True,
        greater_is_better=True,
        evaluation_strategy="epoch",
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        logging_strategy='epoch',
        log_level='info',
        logging_steps=0.1,
        logging_first_step=True,
        save_strategy='epoch',
        save_total_limit=2,
        num_train_epochs=num_epochs,
        auto_find_batch_size=False,
        ignore_data_skip=True,
        disable_tqdm=False,
        overwrite_output_dir=True,
        lr_scheduler_type="cosine",
        fp16_full_eval=False,
        fp16=False,
        fp16_opt_level='O1',
        report_to=['tensorboard']
    )
    return TrainingArguments(**core_params)


def run_training(data, labels, model_name="multitask-classifier", num_epochs=20, holdout_ratio=0.2, **model_args):
    tokenizer = AutoTokenizer.from_pretrained(model_path, cache_dir=cache_dir)
    config = AutoConfig.from_pretrained(model_path, cache_dir=cache_dir)
    train_index, test_index = train_test_split(range(len(data)), test_size=holdout_ratio, random_state=0)
    train_data = data.iloc[train_index]['Comment'].tolist()
    train_labels = labels.iloc[train_index]
    test_data = data.iloc[test_index]['Comment'].tolist()
    test_labels = labels.iloc[test_index]
    eval_dataset = MultitaskDataset(test_data, test_labels, preprocessor=tokenizer)
    model = MultitaskTextClassifier(config, model_path, **model_args)
    core_params = dict(
        model=model,
        args=get_training_args(model_name, num_epochs=num_epochs),
        train_dataset=MultitaskDataset(train_data, train_labels, preprocessor=tokenizer),
        eval_dataset=eval_dataset,
        # compute_metrics=compute_metrics,            
    )
    trainer = Trainer(
        data_collator=lambda x: x[0],
        # callbacks=[transformers.integrations.MLflowCallback],
        **core_params
    )
    # with mlflow.start_run():
    trainer.train()

In [6]:
model_args = {
    "task_labels_map": {ind: len(set(labels.iloc[:, ind].tolist())) for ind in range(labels.shape[1])}
}
run_training(data, labels, **model_args)

***** Running training *****
  Num examples = 1,002
  Num Epochs = 20
  Instantaneous batch size per device = 1
  Total train batch size (w. parallel, distributed & accumulation) = 1
  Gradient Accumulation steps = 1
  Total optimization steps = 20,040
  Number of trainable parameters = 19,225
  0%|          | 0/20040 [00:00<?, ?it/s]

tensor([[-0.1265, -0.1820, -0.3507, -0.5257, -0.4765]], device='cuda:0',
       grad_fn=<ViewBackward0>) tensor([1], device='cuda:0')
tensor([[ 0.0253, -0.0118]], device='cuda:0', grad_fn=<ViewBackward0>) tensor([1], device='cuda:0')
tensor([[ 0.1597, -0.0930,  0.0638,  0.1981,  0.4950]], device='cuda:0',
       grad_fn=<ViewBackward0>) tensor([2], device='cuda:0')
tensor([[ 0.6230, -0.6788, -0.0939,  0.2317,  0.2591]], device='cuda:0',
       grad_fn=<ViewBackward0>) tensor([1], device='cuda:0')
tensor([[-0.8980,  0.2636, -0.3198, -0.0245,  0.1130,  0.0342, -0.2473,  0.3993]],
       device='cuda:0', grad_fn=<ViewBackward0>) tensor([2], device='cuda:0')


RuntimeError: Sizes of tensors must match except in dimension 0. Expected size 5 but got size 2 for tensor number 1 in the list.